<a href="https://colab.research.google.com/github/Sofifi01/CPSN_2026./blob/main/Proyecto_I_%E2%80%93_Parte_IV_Sofia_Salgado.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Visualización para ciencia de datos / Proyecto I – Parte IV (Core)

import io
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files
from IPython.display import display


# --------------------------------------------------
# 1. CARGAR LOS DATOS PREPARADOS EN LA PARTE III
# --------------------------------------------------

print("Selecciona ventas_transformadas.csv.")
archivos = files.upload()

if len(archivos) != 1:
    raise ValueError("Selecciona un solo archivo CSV.")

nombre_archivo = list(archivos.keys())[0]

if not nombre_archivo.lower().endswith(".csv"):
    raise ValueError("El archivo debe tener extensión .csv.")

datos = pd.read_csv(io.BytesIO(archivos[nombre_archivo]))
datos.columns = datos.columns.str.strip()

columnas_necesarias = ["Producto", "Tienda", "Ventas", "Precio"]

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in datos.columns
]

if faltantes:
    raise ValueError(
        f"Faltan estas columnas: {', '.join(faltantes)}. "
        "Utiliza los datos preparados en la Parte III."
    )

# Revisar las columnas numéricas después de leer el CSV.
for columna in ["Ventas", "Precio"]:
    datos[columna] = pd.to_numeric(datos[columna], errors="coerce")
    datos[columna] = datos[columna].replace(
        [float("inf"), float("-inf")],
        float("nan")
    )

if datos.empty:
    raise ValueError("El archivo no contiene registros.")

if datos[columnas_necesarias].isna().any().any():
    raise ValueError(
        "Hay datos esenciales faltantes o no numéricos. "
        "Revisa la limpieza de la Parte III."
    )

if (
    (datos["Ventas"] < 0)
    | (datos["Ventas"] % 1 != 0)
    | (datos["Precio"] < 0)
).any():
    raise ValueError(
        "Hay cantidades o precios inválidos. "
        "Este análisis espera unidades enteras no negativas y precios no negativos."
    )

# Recalcular el ingreso para mantener la misma definición.
datos["Ingreso_Total"] = datos["Ventas"] * datos["Precio"]

# Los gráficos y las tablas se guardarán en esta carpeta.
carpeta = Path("resultados_parte_iv")
carpeta.mkdir(exist_ok=True)

interpretaciones = []


def comentar(texto):
    """Mostrar una interpretación y guardarla para el informe."""
    print(texto)
    interpretaciones.append(texto)


print("\nPRIMERAS 10 FILAS")
display(datos.head(10))

print("\nINFORMACIÓN GENERAL")
datos.info()

print("\nFILAS COMPLETAMENTE DUPLICADAS")
print(datos.duplicated().sum())
print("No se eliminan automáticamente: podrían ser ventas distintas.")


# --------------------------------------------------
# 2. ESTADÍSTICAS DESCRIPTIVAS
# --------------------------------------------------

variables = ["Ventas", "Precio", "Ingreso_Total"]

estadisticas = datos[variables].describe()

print("\nESTADÍSTICAS DESCRIPTIVAS")
display(estadisticas.round(2))

estadisticas.to_csv(carpeta / "estadisticas_descriptivas.csv")

print("\nVALORES FALTANTES")
print(datos.isna().sum())

comentar(f"Se analizaron {len(datos)} registros de ventas.")
comentar(
    f"Total de unidades vendidas: {datos['Ventas'].sum():,.0f}. "
    f"Ingresos totales: {datos['Ingreso_Total'].sum():,.2f}."
)
comentar("Los ingresos no representan ganancias, porque no conocemos los costos.")


# --------------------------------------------------
# 3. COMBINAR HISTOGRAMAS Y BOXPLOTS
# --------------------------------------------------

def graficar_distribucion(datos, variable, color):
    valores = datos[variable]

    q1 = valores.quantile(0.25)
    q3 = valores.quantile(0.75)
    iqr = q3 - q1

    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    filtro_atipicos = (
        (valores < limite_inferior)
        | (valores > limite_superior)
    )

    atipicos = datos.loc[filtro_atipicos].copy()

    figura, ejes = plt.subplots(
        2,
        1,
        figsize=(10, 7),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]}
    )

    # Histograma: muestra la frecuencia de los valores.
    ejes[0].hist(
        valores,
        bins=15,
        color=color,
        edgecolor="black",
        alpha=0.75
    )

    ejes[0].axvline(
        valores.mean(),
        color="darkred",
        linestyle="--",
        label=f"Media: {valores.mean():.2f}"
    )

    ejes[0].axvline(
        valores.median(),
        color="navy",
        linestyle=":",
        label=f"Mediana: {valores.median():.2f}"
    )

    ejes[0].set_title(f"Distribución de {variable}")
    ejes[0].set_ylabel("Cantidad de registros")
    ejes[0].legend()
    ejes[0].grid(axis="y", alpha=0.25)

    # Boxplot: muestra mediana, cuartiles y posibles valores atípicos.
    ejes[1].boxplot(
        valores,
        vert=False,
        whis=1.5,
        patch_artist=True,
        boxprops={"facecolor": color, "alpha": 0.75},
        medianprops={"color": "darkred", "linewidth": 2},
        flierprops={
            "marker": "o",
            "markerfacecolor": "orange",
            "markeredgecolor": "black"
        }
    )

    ejes[1].set_xlabel(variable)
    ejes[1].set_yticks([])
    ejes[1].set_title("Boxplot: regla de 1.5 × IQR")

    figura.tight_layout()
    figura.savefig(
        carpeta / f"distribucion_{variable}.png",
        dpi=150,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(figura)

    print(f"\nINTERPRETACIÓN: {variable}")

    comentar(
        f"{variable}: media = {valores.mean():.2f}, "
        f"mediana = {valores.median():.2f}. "
        f"El intervalo entre cuartiles va de {q1:.2f} a {q3:.2f}."
    )

    asimetria = valores.skew()

    if pd.notna(asimetria):
        if asimetria > 0.5:
            forma = "asimetría hacia valores altos"
        elif asimetria < -0.5:
            forma = "asimetría hacia valores bajos"
        else:
            forma = "asimetría pequeña"

        comentar(
            f"{variable}: {forma}, con coeficiente {asimetria:.2f}. "
            "Se utiliza ±0.5 como umbral orientativo."
        )

    comentar(
        f"{variable}: se detectaron {len(atipicos)} registros "
        "fuera de los límites de 1.5 × IQR."
    )

    if not atipicos.empty:
        display(
            atipicos[
                ["Producto", "Tienda", "Ventas", "Precio", "Ingreso_Total"]
            ].head(10)
        )

    # Guardar todos los atípicos para revisión, sin eliminarlos.
    atipicos.to_csv(
        carpeta / f"atipicos_{variable}.csv",
        index=False
    )


graficar_distribucion(datos, "Ventas", "skyblue")
graficar_distribucion(datos, "Precio", "lightgreen")
graficar_distribucion(datos, "Ingreso_Total", "salmon")

comentar(
    "Los valores atípicos se conservaron: ser poco frecuente "
    "no significa que una venta sea incorrecta."
)


# --------------------------------------------------
# 4. TENDENCIAS DE VENTAS A LO LARGO DEL TIEMPO
# --------------------------------------------------

if "Fecha" in datos.columns:
    datos["Fecha"] = pd.to_datetime(datos["Fecha"], errors="coerce")

    fechas_invalidas = datos["Fecha"].isna().sum()

    comentar(
        f"Registros sin fecha válida excluidos del análisis temporal: "
        f"{fechas_invalidas}."
    )

    datos_temporales = datos.dropna(subset=["Fecha"]).copy()

    if datos_temporales.empty:
        comentar("No hay fechas válidas para construir la tendencia temporal.")

    else:
        # Quitar la hora para agrupar todas las ventas de un mismo día.
        datos_temporales["Fecha"] = datos_temporales["Fecha"].dt.normalize()

        ventas_diarias = datos_temporales.groupby("Fecha").agg(
            Unidades=("Ventas", "sum"),
            Ingresos=("Ingreso_Total", "sum"),
            Registros=("Ventas", "count")
        )

        # Mostrar días sin registros como huecos.
        # No asumimos que la falta de registros significa cero ventas.
        calendario = pd.date_range(
            ventas_diarias.index.min(),
            ventas_diarias.index.max(),
            freq="D"
        )

        ventas_diarias = ventas_diarias.reindex(calendario)
        ventas_diarias.index.name = "Fecha"

        print("\nVENTAS DIARIAS")
        display(ventas_diarias.head(10))

        figura, ejes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

        ejes[0].plot(
            ventas_diarias.index,
            ventas_diarias["Unidades"],
            color="steelblue",
            linewidth=1.5
        )
        ejes[0].set_title("Evolución diaria de las unidades vendidas")
        ejes[0].set_ylabel("Unidades")
        ejes[0].grid(alpha=0.25)

        ejes[1].plot(
            ventas_diarias.index,
            ventas_diarias["Ingresos"],
            color="seagreen",
            linewidth=1.5
        )
        ejes[1].set_title("Evolución diaria de los ingresos")
        ejes[1].set_xlabel("Fecha")
        ejes[1].set_ylabel("Ingresos")
        ejes[1].grid(alpha=0.25)

        figura.autofmt_xdate()
        figura.tight_layout()
        figura.savefig(
            carpeta / "tendencias_diarias.png",
            dpi=150,
            bbox_inches="tight"
        )
        plt.show()
        plt.close(figura)

        ingresos_validos = ventas_diarias["Ingresos"].dropna()
        dias_mayores = ingresos_validos[
            ingresos_validos == ingresos_validos.max()
        ].index

        comentar(
            "Día o días con mayor ingreso registrado: "
            + ", ".join(fecha.strftime("%Y-%m-%d") for fecha in dias_mayores)
            + f", con {ingresos_validos.max():,.2f} cada uno."
        )

        comentar(
            f"Días sin registros dentro del período: "
            f"{ventas_diarias['Registros'].isna().sum()}. "
            "Se muestran como huecos, no como ventas cero."
        )

        comentar(
            "Los picos diarios describen variaciones observadas; "
            "por sí solos no demuestran una tendencia sostenida ni estacionalidad."
        )

        ventas_diarias.to_csv(carpeta / "ventas_diarias.csv")

else:
    comentar(
        "No se generaron gráficos temporales porque falta la columna Fecha. "
        "Para completar esta sección se necesita la fecha real de cada venta."
    )


# --------------------------------------------------
# 5. GRÁFICOS DE DISPERSIÓN
# --------------------------------------------------

figura, ejes = plt.subplots(1, 2, figsize=(13, 5))

# Precio y unidades: explorar si existe asociación.
ejes[0].scatter(
    datos["Precio"],
    datos["Ventas"],
    color="darkorange",
    alpha=0.5
)
ejes[0].set_title("Precio y unidades vendidas")
ejes[0].set_xlabel("Precio unitario")
ejes[0].set_ylabel("Unidades vendidas")
ejes[0].grid(alpha=0.25)

# Unidades e ingresos: el ingreso depende de unidades × precio.
ejes[1].scatter(
    datos["Ventas"],
    datos["Ingreso_Total"],
    color="purple",
    alpha=0.5
)
ejes[1].set_title("Unidades vendidas e ingresos")
ejes[1].set_xlabel("Unidades vendidas")
ejes[1].set_ylabel("Ingreso total")
ejes[1].grid(alpha=0.25)

figura.tight_layout()
figura.savefig(
    carpeta / "relaciones_entre_variables.png",
    dpi=150,
    bbox_inches="tight"
)
plt.show()
plt.close(figura)

correlaciones = datos[variables].corr()

print("\nCORRELACIONES LINEALES DE PEARSON")
display(correlaciones.round(3))

correlacion_precio = correlaciones.loc["Precio", "Ventas"]

if pd.notna(correlacion_precio):
    comentar(
        f"La correlación entre precio y unidades vendidas es "
        f"{correlacion_precio:.3f}. "
        "Un valor cercano a cero indica poca asociación lineal."
    )
else:
    comentar(
        "No se pudo calcular la correlación entre precio y ventas "
        "por falta de variación o de suficientes registros."
    )

comentar(
    "Una correlación no demuestra causalidad. "
    "Además, la relación entre unidades e ingresos depende de la fórmula "
    "Ingreso_Total = Ventas × Precio."
)

correlaciones.to_csv(carpeta / "correlaciones.csv")


# --------------------------------------------------
# 6. COMPARAR PRODUCTOS Y TIENDAS
# --------------------------------------------------

resumen_productos = datos.groupby("Producto").agg(
    Unidades=("Ventas", "sum"),
    Ingresos=("Ingreso_Total", "sum"),
    Registros=("Ventas", "count")
).sort_values("Ingresos", ascending=False)

resumen_tiendas = datos.groupby("Tienda").agg(
    Unidades=("Ventas", "sum"),
    Ingresos=("Ingreso_Total", "sum"),
    Registros=("Ventas", "count")
).sort_values("Ingresos", ascending=False)

print("\nRESUMEN POR PRODUCTO")
display(resumen_productos.round(2))

print("\nRESUMEN POR TIENDA")
display(resumen_tiendas.round(2))

comentar(
    "Los totales por producto y tienda dependen también de cuántos "
    "registros y días de operación tiene cada grupo."
)

resumen_productos.to_csv(carpeta / "resumen_productos.csv")
resumen_tiendas.to_csv(carpeta / "resumen_tiendas.csv")


# --------------------------------------------------
# 7. GUARDAR LAS INTERPRETACIONES
# --------------------------------------------------

with open(
    carpeta / "interpretaciones.txt",
    "w",
    encoding="utf-8"
) as archivo:
    archivo.write("PROYECTO I - PARTE IV\n\n")
    archivo.write("\n\n".join(interpretaciones))

print("\nAnálisis finalizado.")
print("Los gráficos, tablas e interpretaciones están en:", carpeta)
print("Puedes descargarlos desde el panel de archivos de Colab.")


# --------------------------------------------------
# 8. CREAR LA RAMA Y EL PULL REQUEST EN GITHUB
# --------------------------------------------------

# Estos pasos se realizan en tu repositorio local.
# No se ejecutan automáticamente desde este cuaderno.
#
# 1. Guarda el cuaderno como notebooks/proyecto_parte_iv.ipynb.
# 2. Copia resultados_parte_iv/ dentro de tu repositorio.
# 3. Actualiza el README con una descripción de esta etapa.
# 4. Desde la terminal del repositorio, ejecuta:
#
# git switch main
# git pull origin main
# git switch -c analisis-parte-iv
# git add notebooks/proyecto_parte_iv.ipynb resultados_parte_iv/ README.md
# git commit -m "Agregar EDA y visualizaciones de la Parte IV"
# git push -u origin analisis-parte-iv
#
# 5. Abre tu repositorio en GitHub y selecciona Compare & pull request.
# 6. Selecciona main como base y analisis-parte-iv como rama de cambios.
# 7. Describe el análisis y crea el Pull Request para revisión.
#
# No incluyas credenciales ni datos privados en los archivos del repositorio.
